# 🎴 LOR — Sistema de Recomendación de Mazos

Dado un deck_code de entrada, el sistema:
1. Decodifica el mazo a su lista de cartas
2. Lo vectoriza usando el **mismo espacio de features** del clustering original
3. Lo asigna al cluster Ward cuyo centroide está más cercano (nearest centroid)
4. Devuelve el mazo del cluster con mayor **score ponderado por elo**

**Archivo requerido:** `lor_dataset_clustered.csv` (con las columnas `cluster_hier`, `deck_code`, `rank`, `factions`, `card_list`).

---
## 📦 Celda 1 — Imports y configuración

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, fcluster

warnings.filterwarnings('ignore')
np.random.seed(42)

CLUSTER_COLORS = ['#E63946', '#457B9D', '#2A9D8F']

print('✅ Imports OK')

---
## 🔓 Celda 2 — Decodificador de deck codes

Idéntico al usado en `lor_extractor.py`.

In [ ]:
B32 = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ234567'
FACTION_ID = {
    0:'DE', 1:'FR', 2:'IO', 3:'NX', 4:'PZ',
    5:'SI', 6:'BW', 7:'SH', 9:'MT', 10:'BC', 12:'RU',
}
FACTION_NAME = {
    'DE':'Demacia',      'FR':'Freljord',        'IO':'Ionia',
    'NX':'Noxus',        'PZ':'Piltover & Zaun', 'SI':'Shadow Isles',
    'BW':'Bilgewater',   'SH':'Shurima',         'MT':'Mount Targon',
    'BC':'Bandle City',  'RU':'Runeterra',
}

def _varint(data, idx):
    r, s = 0, 0
    while idx < len(data):
        b = data[idx]; idx += 1
        r |= (b & 0x7F) << s
        if not (b & 0x80): break
        s += 7
    return r, idx

def decode(code):
    """Decodifica un deck_code → lista de {card_code, count, set, faction}."""
    bits = ''.join(format(B32.index(c), '05b') for c in code.upper())
    raw  = [int(bits[i:i+8], 2) for i in range(0, (len(bits)//8)*8, 8)]
    cards, idx = [], 1

    for count in (3, 2):
        if idx >= len(raw): break
        ng, idx = _varint(raw, idx)
        for _ in range(ng):
            if idx >= len(raw): break
            nc, idx = _varint(raw, idx)
            if idx >= len(raw): break
            s,  idx = _varint(raw, idx)
            if idx >= len(raw): break
            f,  idx = _varint(raw, idx)
            fc = FACTION_ID.get(f, f'F{f:02d}')
            for _ in range(nc):
                if idx >= len(raw): break
                n, idx = _varint(raw, idx)
                cards.append({'card_code': f'{s:02d}{fc}{n:03d}',
                              'count': count, 'set': s,
                              'faction': FACTION_NAME.get(fc, fc),
                              'faction_code': fc})

    if idx < len(raw):
        ns, idx = _varint(raw, idx)
        for _ in range(ns):
            if idx + 2 >= len(raw): break
            s,  idx = _varint(raw, idx)
            f,  idx = _varint(raw, idx)
            n,  idx = _varint(raw, idx)
            fc = FACTION_ID.get(f, f'F{f:02d}')
            cards.append({'card_code': f'{s:02d}{fc}{n:03d}',
                          'count': 1, 'set': s,
                          'faction': FACTION_NAME.get(fc, fc),
                          'faction_code': fc})
    return cards

# Prueba rápida
test = decode('CECQCAIADIAQQAAGAEEQEOQCAEBBGIACAYBCGJQHAEAQEM')
print(f'✅ Decoder OK — prueba: {len(test)} cartas únicas, '
      f'{sum(c["count"] for c in test)} copias')

---
## 📂 Celda 3 — Carga del dataset entrenado y reconstrucción del espacio de features

Importante: el espacio de features debe ser **exactamente el mismo** del clustering original. Reconstruimos el universo de cartas y facciones desde el dataset clustered.

In [ ]:
df = pd.read_csv('lor_dataset_clustered.csv')
# El dataset clustered no incluye card_list — decodificamos desde deck_code
df['card_list_parsed'] = df['deck_code'].apply(
    lambda code: [{'c': c['card_code'], 'n': c['count']} for c in decode(code)]
)
df['rank'] = pd.to_numeric(df['rank'], errors='coerce')
df['lp']   = pd.to_numeric(df['lp'],   errors='coerce')

print(f'Dataset cargado: {len(df)} mazos')
print(f'Clusters Ward  : {df["cluster_hier"].value_counts().sort_index().to_dict()}')

# Reconstruir el universo de cartas y facciones exactamente igual
KNOWN_FACTIONS = [
    'Demacia', 'Freljord', 'Ionia', 'Noxus', 'Piltover & Zaun',
    'Shadow Isles', 'Bilgewater', 'Shurima', 'Mount Targon',
    'Bandle City', 'Runeterra',
]
card_universe = sorted({c['c'] for cl in df['card_list_parsed'] for c in cl})
unknown_facs  = sorted({
    f.strip()
    for facs in df['factions']
    for f in facs.split('|')
    if f.strip() not in KNOWN_FACTIONS and f.strip()
})
all_factions = KNOWN_FACTIONS + unknown_facs

card_idx    = {c: i for i, c in enumerate(card_universe)}
faction_idx = {f: i for i, f in enumerate(all_factions)}
n_cards, n_fac = len(card_universe), len(all_factions)

print(f'\nEspacio de features reconstruido:')
print(f'  Cartas únicas    : {n_cards}')
print(f'  Facciones        : {n_fac}')
print(f'  Total dimensiones: {n_cards + n_fac}')

---
## 🔢 Celda 4 — Vectorización y cálculo de centroides por cluster

Aplicamos la misma vectorización del clustering original a todo el dataset y calculamos el centroide de cada cluster en el espacio PCA.

In [ ]:
def vectorize_deck(card_list, factions_str):
    """Convierte un mazo en el vector de features del espacio del clustering."""
    vec = np.zeros(n_cards + n_fac, dtype=np.float64)
    for c in card_list:
        col = card_idx.get(c['c'])
        if col is not None:
            vec[col] = c['n']
    for f in factions_str.split('|'):
        col = faction_idx.get(f.strip())
        if col is not None:
            vec[n_cards + col] = 1.0
    return vec

# Matriz del dataset entrenado
X_train = np.array([
    vectorize_deck(row['card_list_parsed'], row['factions'])
    for _, row in df.iterrows()
])

print(f'Matriz X_train: {X_train.shape}')

# Ajustar el scaler y PCA con los mismos datos que se usaron en el clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

# PCA: retener 95% varianza (igual que en el clustering original)
n_max     = min(X_scaled.shape[0], X_scaled.shape[1])
pca_full  = PCA(n_components=n_max, random_state=42)
pca_full.fit(X_scaled)
cumvar    = np.cumsum(pca_full.explained_variance_ratio_)
n_95      = int(np.searchsorted(cumvar, 0.95)) + 1

pca       = PCA(n_components=n_95, random_state=42)
X_pca     = pca.fit_transform(X_scaled)

print(f'PCA: {n_95} componentes para 95% varianza')
print(f'X_pca shape: {X_pca.shape}')

# Calcular centroides en el espacio PCA por cluster
centroids = {}
for k in sorted(df['cluster_hier'].unique()):
    mask = df['cluster_hier'] == k
    centroids[k] = X_pca[mask].mean(axis=0)

print(f'\nCentroides calculados para {len(centroids)} clusters:')
for k, c in centroids.items():
    n_in_cluster = (df['cluster_hier'] == k).sum()
    print(f'  Cluster {k}: n={n_in_cluster}, centroide ‖·‖₂ = {np.linalg.norm(c):.3f}')

---
## 🏆 Celda 5 — Score ponderado por elo y mejor mazo por cluster

Recomputamos el score del EDA: $\text{prestige}(f) = \sum_{j \in \text{usuarios}(f)} 1/\text{rank}(j)$, normalizado a [0, 1]. El score de un mazo es la suma del prestige normalizado de sus facciones.

In [ ]:
# Expandir facciones por mazo
df['faction_list'] = df['factions'].apply(
    lambda x: [f.strip() for f in str(x).split('|') if f.strip()]
)
df_fac = df.explode('faction_list').rename(columns={'faction_list': 'faction'})
df_fac = df_fac[df_fac['faction'] != ''].copy()

# Prestige por facción
faction_prestige = (
    df_fac.assign(weight=lambda x: 1 / x['rank'])
          .groupby('faction')['weight'].sum()
)
fp_norm = (
    (faction_prestige - faction_prestige.min()) /
    (faction_prestige.max() - faction_prestige.min())
)

# Score por mazo
def deck_score(faction_list):
    return sum(fp_norm.get(f, 0.0) for f in faction_list)

df['score']      = df['faction_list'].apply(deck_score)
df['score_norm'] = (
    (df['score'] - df['score'].min()) /
    (df['score'].max() - df['score'].min())
).round(4)

# Mejor mazo por cluster (mayor score_norm; empate → menor rank)
best_by_cluster = {}
for k in sorted(df['cluster_hier'].unique()):
    subset = df[df['cluster_hier'] == k].sort_values(
        ['score_norm', 'rank'], ascending=[False, True]
    )
    best_by_cluster[k] = subset.iloc[0]

print('Mejor mazo por cluster (criterio: score_norm más alto):\n')
summary = pd.DataFrame([
    {
        'cluster':     k,
        'n_mazos':     (df['cluster_hier'] == k).sum(),
        'player':      row['player_name'],
        'rank':        int(row['rank']),
        'lp':          row['lp'],
        'score_norm':  row['score_norm'],
        'factions':    row['factions'][:50] + ('...' if len(row['factions']) > 50 else ''),
    }
    for k, row in best_by_cluster.items()
])
display(summary)

print('\nDeck codes de los mejores mazos por cluster:')
for k, row in best_by_cluster.items():
    print(f'  Cluster {k}: {row["deck_code"]}')

---
## 🎯 Celda 6 — Función de recomendación

Toma un deck_code, lo asigna al cluster del centroide más cercano (distancia euclidiana en el espacio PCA) y devuelve el mejor mazo del cluster.

In [ ]:
def recommend(input_deck_code: str, verbose: bool = True) -> dict:
    """
    Asigna un mazo nuevo a su cluster (nearest centroid) y retorna el mejor mazo.

    Returns dict con:
      input_deck_code      : código de entrada
      input_factions       : facciones detectadas en el mazo de entrada
      input_total_cards    : total de copias decodificadas
      cluster_assigned     : cluster asignado por nearest centroid
      distance_to_centroid : distancia al centroide del cluster ganador
      coverage_pct         : % de copias del mazo cubiertas por el universo de cartas conocido
      recommended_deck     : deck_code recomendado (mejor del cluster)
      recommended_player   : nombre del jugador del mazo recomendado
      recommended_rank     : rank de ese jugador
      recommended_score    : score_norm del mazo recomendado
    """
    # 1. Decodificar
    try:
        cards = decode(input_deck_code)
    except Exception as e:
        return {'error': f'Error decodificando: {e}'}

    if not cards:
        return {'error': 'Mazo vacío o código inválido'}

    total = sum(c['count'] for c in cards)
    facs  = sorted({c['faction'] for c in cards})

    # 2. Vectorizar usando el mismo espacio
    card_list_fmt = [{'c': c['card_code'], 'n': c['count']} for c in cards]
    factions_str  = '|'.join(facs)
    vec_raw       = vectorize_deck(card_list_fmt, factions_str)

    # Cobertura: cuántas copias del mazo tienen una dimensión asignada
    covered_copies = sum(
        c['count']
        for c in cards
        if c['card_code'] in card_idx
    )
    coverage = covered_copies / total if total > 0 else 0

    # 3. Proyectar al espacio PCA (usar transform, NO fit_transform)
    vec_scaled = scaler.transform(vec_raw.reshape(1, -1))
    vec_pca    = pca.transform(vec_scaled)[0]

    # 4. Asignar al cluster del centroide más cercano
    distances = {
        k: np.linalg.norm(vec_pca - centroid)
        for k, centroid in centroids.items()
    }
    cluster_assigned = min(distances, key=distances.get)

    # 5. Recuperar el mejor mazo de ese cluster
    best = best_by_cluster[cluster_assigned]

    result = {
        'input_deck_code':      input_deck_code,
        'input_factions':       facs,
        'input_total_cards':    total,
        'cluster_assigned':     int(cluster_assigned),
        'distance_to_centroid': round(distances[cluster_assigned], 4),
        'all_distances':        {int(k): round(v, 4) for k, v in distances.items()},
        'coverage_pct':         round(coverage * 100, 1),
        'recommended_deck':     best['deck_code'],
        'recommended_player':   best['player_name'],
        'recommended_rank':     int(best['rank']),
        'recommended_score':    float(best['score_norm']),
        'recommended_factions': best['factions'],
    }

    if verbose:
        print(f'  Mazo entrada     : {input_deck_code[:50]}...')
        print(f'  Cartas decodif.  : {total}  |  Facciones: {facs}')
        print(f'  Cobertura        : {result["coverage_pct"]}% (cartas conocidas)')
        print(f'  Distancias       : {result["all_distances"]}')
        print(f'  → Cluster asignado: {result["cluster_assigned"]} '
              f'(dist={result["distance_to_centroid"]})')
        print(f'  → Mazo recomend. : {result["recommended_deck"]}')
        print(f'    Jugador        : {result["recommended_player"]} '
              f'(rank {result["recommended_rank"]}, score {result["recommended_score"]})')

    return result

# Validación: pasar uno del dataset y verificar que asigna su mismo cluster
print('Validación con un mazo del dataset:')
test_row = df.iloc[0]
test_result = recommend(test_row['deck_code'], verbose=False)
print(f'  Mazo: {test_row["player_name"]}')
print(f'  Cluster real    : {test_row["cluster_hier"]}')
print(f'  Cluster asignado: {test_result["cluster_assigned"]}')
match = '✅ coincide' if test_row['cluster_hier'] == test_result['cluster_assigned'] else '⚠️ NO coincide'
print(f'  {match}')

---
## 🆚 Celda 7 — Aplicación a la lista de mazos de rivales

In [ ]:
RIVAL_DECKS = [
    'CQCACBQCBIBAKCR2WYAQEBICC4NAIAICBAHSYMIFAECAEDYBAUFNKAIBAUBBKAIGAIUAEAICBMXACAIIBIDA',
    'CEBACBQBDUCACAISEYUCSBABAEBCSAIDAELACCABDACACAIBBMLB4CIBAEATOAICAECACAYBAIAQGAQIAEDACEQBAYBCAAIIAEJQEAICBMYQEBABBIGA',
    'CQBACBQKF4AQMCJIA4AQGCJNAECQUHIBA4EQOAIIAUJAECAJAMDAKBIJAIBQKBQNAYDASBQKEESSSKYA',
    'CQCACAIAGIAQQBISAEEQUAICBEAB6IQFAEAQKAIBAQAAMAIJAUGAECIKCIKAECIADYSAKAIBAAQACAIFB4AQOBICAIEQKDQPAYEQAIJHFAUSWLI',
    'CECQCBACBYAQMAQJAIAQEBQJAMAQKFIXGECQMBIMBYIBIJQCAEBQEFABAYCQ2AA',
    'CEBAGBQAAYKSEBIGAUGBAFBGFYCQCAIFGEAQIAADAEDAAGABA4AAKAYGAUBQ2DQCAEDAAHABAYCQO',
    'CEBQCCADBEAQQBJKAICAKNRXAYAQIBIDAEEQKDICBABRSHADAECQCBBRAMDAKJRNFYBQQBIHEUUQEAIBAUHQCBQFA4',
    'CUCQCAIFGEAQOBICAEEAKFICAYFBUHYCAYER2LYFAEAQKIABAYGACAIGAUWQCCIFBUBAQBISFIBQCCAFFEBAIBIQCECACBIBAMOSC',
    'CQCQCAIFAEAQKBIDAEDAKLQCAQCTKNYCBACQOKQGAEDAKLIBBABRYAIJBIKAEAIFEAYQEBAFAQ3AECAFCIUQCAIEAU4A',
    'CEBQCCABCMBAKBACAYCACBAIDE2DUBYBAMAQEAIDAQFQCBABBIAQOBARAEEAIGACAEAQIKQCAUCAUDQBAEAQCHQ',
    'CQCQCCAKAUAQQBAYAIDQUCIUAMCQIAQGBICACBAIDE2DUAYBAUFEMAIFAQHACBYECEAQCBIKUQAQ',
    'CUCQCAIECQAQGBATAEDAUKYBA4GA6AQBAEEBABIBAEASWAIBAMRACCIBBIAQSBYMAMCAOMJ3NUEACAIAGEAQCAQFAEAQKJIBAIDAUAICAEBQCAYJE4AQKCUMAEAQMAIV',
    'CEBACCABDABACAJGE4DACAIFAEAQGAICAECACCQBA4CQEAIIAEJQMAIBAEFRMKBJG4DACAIFB4AQGAIWAEEAABQCAEAREHQCAQAQWEYCAYASIJQ',
    'CIAAMAIFBEDACBYAAUBAIAAQCQBQCAANDI3QKAYAAIBQMCALAYBQSAYOEYZUQVYCAIBQAAIHAIBQSAQG',
    'CIBQCBYGFMBQMBQGCAOAGCAGAIEAUAYCBADA6FYDBACAMCITAQDAMCY6EMUACAIIAYEQ',
]

print(f'Procesando {len(RIVAL_DECKS)} mazos de rivales...\n')

results = []
for i, code in enumerate(RIVAL_DECKS, 1):
    print(f'━━━ Rival {i}/{len(RIVAL_DECKS)} ━━━')
    r = recommend(code, verbose=True)
    results.append(r)
    print()

---
## 📋 Celda 8 — Tabla resumen de recomendaciones

In [ ]:
df_results = pd.DataFrame([
    {
        'rival_n':            i,
        'input_factions':     '|'.join(r['input_factions']),
        'cards_decoded':      r['input_total_cards'],
        'coverage_pct':       r['coverage_pct'],
        'cluster_assigned':   r['cluster_assigned'],
        'dist_centroid':      r['distance_to_centroid'],
        'recommended_player': r['recommended_player'],
        'recommended_rank':   r['recommended_rank'],
        'recommended_score':  r['recommended_score'],
        'recommended_deck':   r['recommended_deck'][:30] + '...',
    }
    for i, r in enumerate(results, 1)
    if 'error' not in r
])

print('Resumen de recomendaciones para los 15 rivales:\n')
display(df_results)

print(f'\nDistribución de clusters asignados a rivales:')
print(df_results['cluster_assigned'].value_counts().sort_index()
      .rename(lambda x: f'Cluster {x}').to_string())

print(f'\nCobertura media de los mazos de rivales: '
      f'{df_results["coverage_pct"].mean():.1f}%')
print(f'(porcentaje de cartas reconocidas del universo del leaderboard Maestro)')

---
## 🗺️ Celda 9 — Visualización: rivales sobre el espacio PCA 2D

In [ ]:
# PCA 2D solo para visualización
pca_2d   = PCA(n_components=2, random_state=42)
X_2d     = pca_2d.fit_transform(X_scaled)
var1, var2 = pca_2d.explained_variance_ratio_ * 100

# Proyectar también los rivales
rival_2d = []
for code in RIVAL_DECKS:
    try:
        cards = decode(code)
        cl    = [{'c': c['card_code'], 'n': c['count']} for c in cards]
        facs  = '|'.join(sorted({c['faction'] for c in cards}))
        v     = vectorize_deck(cl, facs).reshape(1, -1)
        v_s   = scaler.transform(v)
        v_2d  = pca_2d.transform(v_s)[0]
        rival_2d.append(v_2d)
    except:
        rival_2d.append([np.nan, np.nan])
rival_2d = np.array(rival_2d)

fig, ax = plt.subplots(figsize=(11, 7))

# Mazos del dataset (entrenamiento), coloreados por cluster
for k in sorted(df['cluster_hier'].unique()):
    mask = (df['cluster_hier'] == k).values
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=CLUSTER_COLORS[k], s=55, alpha=0.55,
               edgecolors='white', linewidths=0.5,
               label=f'Cluster {k} entrenamiento (n={mask.sum()})',
               zorder=2)

# Centroides como estrellas grandes
for k, c in centroids.items():
    # Proyectar el centroide del espacio PCA n_95 a PCA 2D para mostrarlo
    # Lo más correcto: tomar la media de los puntos en X_2d
    mask = (df['cluster_hier'] == k).values
    cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
    ax.scatter(cx, cy, c=CLUSTER_COLORS[k], marker='*',
               s=400, edgecolors='black', linewidths=1.2,
               zorder=4, label=f'Centroide C{k}')

# Rivales como triángulos
valid_rivals = ~np.isnan(rival_2d[:, 0])
for i, (xy, valid) in enumerate(zip(rival_2d, valid_rivals)):
    if not valid: continue
    cluster_r = results[i]['cluster_assigned']
    ax.scatter(xy[0], xy[1], marker='^',
               c=CLUSTER_COLORS[cluster_r], s=160,
               edgecolors='black', linewidths=1.0, zorder=5)
    ax.annotate(f'R{i+1}', xy=xy, xytext=(6, 4),
                textcoords='offset points',
                fontsize=8, fontweight='bold', zorder=6)

ax.set_xlabel(f'PC1 ({var1:.1f}%)')
ax.set_ylabel(f'PC2 ({var2:.1f}%)')
ax.set_title('Rivales asignados a clusters por nearest centroid\n'
             '(△ = rival, ● = mazo del leaderboard Maestro, ★ = centroide)',
             fontweight='bold')
ax.axhline(0, color='gray', lw=0.4, ls='--')
ax.axvline(0, color='gray', lw=0.4, ls='--')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lor_rivales_pca.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Celda 10 — Exportar resultados

In [ ]:
# Tabla completa de resultados
df_full = pd.DataFrame([
    {
        'rival_n':              i,
        'input_deck_code':      r['input_deck_code'],
        'input_factions':       '|'.join(r['input_factions']),
        'input_total_cards':    r['input_total_cards'],
        'coverage_pct':         r['coverage_pct'],
        'cluster_assigned':     r['cluster_assigned'],
        'dist_centroid':        r['distance_to_centroid'],
        'recommended_deck':     r['recommended_deck'],
        'recommended_player':   r['recommended_player'],
        'recommended_rank':     r['recommended_rank'],
        'recommended_score':    r['recommended_score'],
        'recommended_factions': r['recommended_factions'],
    }
    for i, r in enumerate(results, 1)
    if 'error' not in r
])
df_full.to_csv('lor_rivales_recomendaciones.csv', index=False)

print('✅ Archivos exportados:')
print('   lor_rivales_recomendaciones.csv')
print('   lor_rivales_pca.png')

import sys
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('lor_rivales_recomendaciones.csv')
    files.download('lor_rivales_pca.png')